# Task-Aware Compression of Sentence Embeddings — Results Analysis

**Course:** NLP with Deep Learning (Sem 6)  
**Team:** Sanyam Verma, Archit Jaju, Pushkar Kulkarni, Harjot Singh  
**Base Encoder:** `sentence-transformers/all-mpnet-base-v2` (768-dim)  
**Compression Methods:** Linear · Autoencoder · Distillation  
**Dimensions:** 32 · 64 · 128 · 256  
**Tasks:** STS-B (Spearman ρ) · SNLI (Accuracy) · SST-2 (Accuracy)

In [ ]:
import os
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

# Inline display
%matplotlib inline

# Resolve project root whether notebook is run from notebooks/ or project root
_here = os.path.abspath("")
PROJECT_ROOT = os.path.normpath(os.path.join(_here, "..")) \
               if os.path.basename(_here) == "notebooks" \
               else _here

METRICS_DIR  = os.path.join(PROJECT_ROOT, "results", "metrics")
PLOTS_DIR    = os.path.join(PROJECT_ROOT, "results", "plots")

print(f"Project root : {PROJECT_ROOT}")
print(f"Metrics dir  : {METRICS_DIR}")
print(f"Plots dir    : {PLOTS_DIR}")

## 2. Load Results CSV

In [ ]:
csv_path = os.path.join(METRICS_DIR, "full_results_table.csv")
df = pd.read_csv(csv_path)

print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}\n")
df.head(10)

## 3. Baseline Scores (768-dim, No Compression)

In [ ]:
baseline = pd.DataFrame({
    "Task":    ["STS-B",       "SNLI",        "SST-2 (CLS)"],
    "Metric":  ["Spearman ρ",  "Accuracy",    "Accuracy"],
    "Score":   [0.8275,        0.7671,         0.8865],
    "Encoder": ["all-mpnet-base-v2 (768-dim)"] * 3,
})
baseline = baseline.set_index("Task")
print("Baseline performance — 768-dim uncompressed embeddings:\n")
display(baseline.style.format({"Score": "{:.4f}"}))

BASELINES = {"STS_spearman": 0.8275, "NLI_accuracy": 0.7671, "CLS_accuracy": 0.8865}

## 4. Top-5 Best Performing Configurations Per Task

In [ ]:
task_configs = [
    ("STS-B",        "STS_spearman",  0.8275),
    ("SNLI",         "NLI_accuracy",  0.7671),
    ("SST-2 (CLS)",  "CLS_accuracy",  0.8865),
]

# Exclude baseline row for ranking
ranked = df[df["Mode"] != "baseline"].copy()

for task_name, metric_col, base_score in task_configs:
    top5 = (ranked
            .nlargest(5, metric_col)
            [["Method", "Mode", "TrainTask", "Dim", metric_col]]
            .reset_index(drop=True))
    top5.index += 1
    top5["vs_baseline"] = (top5[metric_col] - base_score).map("{:+.4f}".format)
    print(f"\n{'─'*60}")
    print(f"  {task_name}  (baseline = {base_score})")
    print(f"{'─'*60}")
    display(top5.style.format({metric_col: "{:.4f}"}))

## 5. Worst Cross-Task Generalization Cases

A task-aware compressor trained on task **A** evaluated on task **B ≠ A**.  
Negative delta = performance drop vs the task-agnostic baseline for the same method/dim.

In [ ]:
aware = df[df["Mode"] == "task_aware"].copy()
agnostic = df[df["Mode"] == "task_agnostic"].copy()

metric_map = {"sts": "STS_spearman", "nli": "NLI_accuracy", "classification": "CLS_accuracy"}
rows = []

for _, aw_row in aware.iterrows():
    train_task = aw_row["TrainTask"]
    method     = aw_row["Method"]
    dim        = aw_row["Dim"]

    for eval_task, metric_col in metric_map.items():
        if eval_task == train_task:
            continue  # same-task — skip

        ag_match = agnostic[(agnostic["Method"] == method) & (agnostic["Dim"] == dim)]
        if ag_match.empty:
            continue

        ag_score = ag_match[metric_col].values[0]
        aw_score = aw_row[metric_col]
        delta    = aw_score - ag_score

        rows.append({
            "Method":      method,
            "Dim":         dim,
            "Train Task":  train_task,
            "Eval Task":   eval_task,
            "Aware Score": round(aw_score, 4),
            "Agnostic":    round(ag_score, 4),
            "Δ (aware−agnostic)": round(delta, 4),
        })

cross_df = pd.DataFrame(rows).sort_values("Δ (aware−agnostic)").reset_index(drop=True)
cross_df.index += 1
print("Worst 5 cross-task generalization cases (largest negative delta):")
display(cross_df.head(5).style.format({"Aware Score": "{:.4f}", "Agnostic": "{:.4f}",
                                        "Δ (aware−agnostic)": "{:+.4f}"}))

## 6. All Figures (Fig 1 – Fig 7)

In [ ]:
figures = [
    ("fig1_perf_vs_dim.png",              "Fig 1 — Performance vs Compressed Dimension (Task-Agnostic)"),
    ("fig2_cross_task_heatmap.png",        "Fig 2 — Cross-Task Generalization Heatmap (dim=128)"),
    ("fig3_method_comparison.png",         "Fig 3 — Method Comparison at dim=128: Agnostic vs Aware"),
    ("fig4_aware_vs_agnostic_delta.png",   "Fig 4 — Task-Aware vs Task-Agnostic Score Delta"),
    ("fig5_sentence_length_analysis.png",  "Fig 5 — STS Performance by Sentence Length Bucket"),
    ("fig6_tsne_comparison.png",           "Fig 6 — t-SNE of NLI Embeddings: Raw 768-dim vs Compressed 128-dim"),
    ("fig7_compression_error.png",         "Fig 7 — Compression Fidelity: Mean Cosine Similarity"),
]

for fname, title in figures:
    fpath = os.path.join(PLOTS_DIR, fname)
    if os.path.exists(fpath):
        display(Markdown(f"### {title}"))
        display(Image(filename=fpath, width=900))
    else:
        print(f"  [Missing] {fname} — run scripts/plot_results.py and analysis/linguistic.py first")

## 7. Key Findings Summary Table

In [ ]:
import numpy as np

d128 = df[(df["Dim"] == 128) & (df["Mode"] != "baseline")]
methods = ["linear", "autoencoder", "distillation"]

task_metric_map = {"sts": "STS_spearman", "nli": "NLI_accuracy", "classification": "CLS_accuracy"}
task_label_map  = {"sts": "STS (Spearman ρ)", "nli": "NLI (Acc)", "classification": "CLS (Acc)"}

# ── 1. Best method at dim=128 per task (aware own-task) ──────────────────────
best_rows = []
for task, metric_col in task_metric_map.items():
    subset = d128[(d128["Mode"] == "task_aware") & (d128["TrainTask"] == task)]
    if not subset.empty:
        best = subset.loc[subset[metric_col].idxmax()]
        best_rows.append({
            "Task":          task_label_map[task],
            "Best Method":   best["Method"].title(),
            "Mode":          "Task-Aware",
            "Dim":           128,
            "Score":         round(best[metric_col], 4),
            "vs Baseline":   f"{best[metric_col] - BASELINES[metric_col]:+.4f}",
        })

best_df = pd.DataFrame(best_rows).set_index("Task")
print("Best method at dim=128 — Task-Aware (own task training):")
display(best_df.style.format({"Score": "{:.4f}"}))

# ── 2. Average score retention per method (compressed/baseline) ──────────────
retention_rows = []
for method in methods:
    m_df    = df[(df["Method"] == method) & (df["Mode"] == "task_agnostic")]
    retains = []
    for metric_col, base in BASELINES.items():
        retains.extend((m_df[metric_col] / base).tolist())
    retention_rows.append({
        "Method":                    method.title(),
        "Avg Score Retention (%)":   round(np.nanmean(retains) * 100, 1),
    })

ret_df = pd.DataFrame(retention_rows).set_index("Method")
print("\nAverage score retention vs baseline — Task-Agnostic, all dims:")
display(ret_df)

# ── 3. Cross-task drop per method ─────────────────────────────────────────────
ct_deltas = cross_df.groupby("Method")["Δ (aware−agnostic)"].mean().reset_index()
ct_deltas.columns = ["Method", "Mean Cross-Task Δ"]
ct_deltas["Method"] = ct_deltas["Method"].str.title()
ct_deltas = ct_deltas.sort_values("Mean Cross-Task Δ", ascending=False)
print("\nMean cross-task delta per method (higher = better generalisation):")
display(ct_deltas.set_index("Method").style.format({"Mean Cross-Task Δ": "{:+.4f}"}))